In [ ]:
# ==============================================================================
# 4_evaluation.ipynb
# ==============================================================================
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression as SkLearnLR
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

print("\n>>> STEP 5: Scalability Test (PySpark vs Scikit-Learn)...")

sample_frac = 0.01
pdf_sample = model_data.sample(fraction=sample_frac, seed=42).toPandas()

if not pdf_sample.empty:
    X_sk = pdf_sample["features"].apply(lambda x: x.toArray()).tolist()
    y_sk = pdf_sample["label"].tolist()

    start = time.time()
    try:
        SkLearnLR(max_iter=5).fit(X_sk, y_sk)
        sk_time = time.time() - start
        print(f"   Scikit-Learn Time (on tiny 1% subset): {sk_time:.4f} seconds")
    except Exception as e:
        print(f"   Scikit-Learn Error: {e}")

print("\n>>> STEP 6: Generating Tableau Parquet/CSV Extract...")

if best_predictions is not None:
    # 1. Unpack vectors for Tableau
    df_with_array = best_predictions.withColumn("features_array", vector_to_array("features"))
    df_tableau_prep = df_with_array \
        .withColumn("feature_1", col("features_array")[0]) \
        .withColumn("feature_2", col("features_array")[1]) \
        .withColumn("feature_22", col("features_array")[21])

    tableau_cols = ["label", "prediction", "feature_1", "feature_2", "feature_22"]
    output_path = "/content/drive/MyDrive/Bigdata/HIGGS_tableau_extract.parquet"

    df_tableau_prep.select(*tableau_cols).coalesce(1).write.mode("overwrite").parquet(output_path)
    print(f"SUCCESS: Tableau Parquet file saved at: {output_path}")

    # Convert Parquet to CSV
    print("Converting Parquet to CSV...")
    df_convert = pd.read_parquet(output_path)
    csv_path = "/content/drive/MyDrive/Bigdata/HIGGS_tableau_extract.csv"
    df_convert.to_csv(csv_path, index=False)
    print(f"Tableau file ready {csv_path}")